# Load epub book

In [1]:
# Import libraries
import os
from langchain_community.document_loaders import UnstructuredEPubLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb
from uuid import uuid4
from chromadb.utils import embedding_functions

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

W0521 01:23:23.341000 902 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0521 01:23:23.415000 902 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [2]:
# TODO: Load document
chunk_size=1024
chunk_overlap=100

text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

epub_loader = UnstructuredEPubLoader(file_path='/content/docs/charles-dickens_a-christmas-carol.epub')

In [3]:
# TODO Split document
chunks = epub_loader.load_and_split(text_splitter)
print(len(chunks))

[WARNING] Sandbox argument was used, but pandoc version is too low. Ignoring argument.


204


In [ ]:
# TODO Examine chunk
idx = 50
print(chunks[idx].page_content)
print(chunks[idx].metadata)


It was a strange figure﻿—like a child; yet not so like a child as like an old man, viewed through some supernatural medium, which gave him the appearance of having receded from the view, and being diminished to a child’s proportions. Its hair, which hung about its neck and down its back, was white, as if with age; and yet the face had not a wrinkle in it, and the tenderest bloom was on the skin. The arms were very long and muscular; the hands the same, as if its hold were of uncommon strength. Its legs and feet, most delicately formed, were, like those upper members, bare. It wore a tunic of the purest white; and round its waist was bound a lustrous belt, the sheen of which was beautiful. It held a branch of fresh green holly in its hand; and, in singular contradiction of that wintry emblem, had its dress trimmed with summer flowers. But the strangest thing about it was, that from the crown of its head there sprang a bright clear jet of light, by which all this was visible; and which w

# Create embeddings

In [4]:
# TODO: Create embedding model
embed_model_name = "BAAI/bge-small-en-v1.5"
#embed_model_name = "all-MiniLM-L6-v2"
embed_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=embed_model_name)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# TODO: Explore embedding model
text = [ 'hello, world', 'TorchCodec is a Python library for decoding video and audio data into PyTorch tensors, on CPU and CUDA GPU. It also supports video and audio encoding on CPU! It aims to be fast, easy to use, and well integrated into the PyTorch ecosystem. If you want to use PyTorch to train ML models on videos and audio, TorchCodec is how you turn these into data.']

embeddings = embed_function(text)
print(len(embeddings[0]))
print(len(embeddings[1]))
print(embeddings[0])

384
384
[-3.15819047e-02 -4.86476645e-02  3.21324095e-02 -6.57483488e-02
 -1.12420879e-03  1.14271799e-02 -1.62236346e-03  5.49600758e-02
  4.48704287e-02 -2.09958665e-03  7.87422992e-03 -2.20074505e-02
  3.43550555e-02  6.57045990e-02  2.98711974e-02 -2.77359504e-04
  1.02014421e-03 -3.47684957e-02 -1.21079296e-01 -1.47990528e-02
  9.72586572e-02  3.53694819e-02 -1.68968756e-02 -4.28635441e-02
 -2.48042066e-02  5.63814165e-03  6.80473307e-03  1.35494387e-02
  6.07590657e-03 -9.83635113e-02 -6.45543784e-02 -1.15323728e-02
  3.96090746e-02  2.41095573e-02  4.54738475e-02 -2.10404806e-02
  2.52141319e-02 -1.03886537e-02 -7.94329494e-02  3.64227779e-03
  4.60232794e-02 -5.09504452e-02  1.40664671e-02 -3.41337710e-03
  1.36135798e-02 -4.93411534e-02  1.70673206e-02  5.47222644e-02
 -2.78037693e-02  4.88187245e-04 -5.45994379e-02 -8.51238705e-03
 -1.97878145e-02 -2.24604411e-03  2.84831468e-02  9.09865052e-02
  7.97384828e-02  2.93895905e-03  4.68927957e-02  8.69193859e-03
  1.88648328e-02 

In [5]:
# TODO: Prepare the chunks for inserting into Chroma
texts = [ d.page_content for d in chunks ]
text_ids = [ str(uuid4())[:8] for _ in range(len(texts))]
print(len(texts))
print(len(text_ids))
print(text_ids)


204
204
['fbed57ff', '4799cb6b', '54e673bc', '08fb3374', '15906563', '01e684a0', '8ee5b48b', '3d2f8c8a', '36152174', '91f00aad', '6d1410b5', '28c08a77', 'd30ba696', '9e146420', '31e9763a', 'afac1f89', '5fd05d1e', 'b41a1c9a', 'd571bdb0', '8b7ed762', '74908f56', 'd3f5966d', 'fc88bc8c', '0492d458', 'b39d2f4e', 'f32bb9c6', 'e7512c26', '7c21269f', 'b7d56fb7', '2d3a5350', '2f2990e3', '6d9b1e35', '1f8d618c', '31496e56', '32396b9b', '2de83526', 'e2f0dca6', '9a756496', 'b6d83d59', '1982486b', '49e39cf1', '787124ad', '8a1f0d39', '76b396f9', '69e57676', '7ff31b91', '8bac5c00', 'b4d0a116', '8cee0795', '0d149332', '0ad89b68', '60925da8', '07f986ae', 'cfc14dae', 'b53d1098', '475b45c1', '8849a0aa', '30b80db1', '45ddbf1a', '2f9d72bf', '79c78f4d', 'fba78e78', 'cecc87d1', '849d056a', 'd9f57175', '2c4da479', '58f53ca3', 'f4f90593', '203f7706', 'd6d04097', '66650f86', 'fde1a813', '9de4e6be', 'dec3d30e', 'd72e28fb', '040c4b58', 'b4cb083d', '4d4d40df', '273e5885', '0e038bc0', '15f0fda0', '0b5e7b68', '7ac2d9

In [6]:
# TODO: Create ephemeral Chroma client and save chunks
col_name = 'a-christmas-carol'

ch_client = chromadb.Client()

# clean the database
try:
  ch_client.delete_collection(col_name)
except:
  pass

# create the collection
ch_collection = ch_client.create_collection(
    name=col_name,
    embedding_function=embed_function
)


In [7]:
# Insert the documents into the database
ch_collection.add(
    documents=texts,
    ids=text_ids
)

In [8]:
# TODO: Print number of documents in collection
print(ch_collection.count())

204


In [9]:
# TODO: Query collection
query = "The ghost of christmas past is the first ghost to appear before Scrooge"

results = ch_collection.query(
    query_texts=query,
    n_results=5
)

for k, v in results.items():
  print(f'{k}: {v}')

ids: [['ff211225', 'f363d433', '70c7f908', '07f986ae', '8557a518']]
embeddings: None
documents: [['“Come in!” exclaimed the Ghost. “Come in! and know me better, man!”\n\nScrooge entered timidly, and hung his head before this Spirit. He was not the dogged Scrooge he had been; and though the Spirit’s eyes were clear and kind, he did not like to meet them.\n\n“I am the Ghost of Christmas Present,” said the Spirit. “Look upon me!”', '“I am the Ghost of Christmas Present,” said the Spirit. “Look upon me!”\n\nScrooge reverently did so. It was clothed in one simple deep green robe, or mantle, bordered with white fur. This garment hung so loosely on the figure, that its capacious breast was bare, as if disdaining to be warded or concealed by any artifice. Its feet, observable beneath the ample folds of the garment, were also bare; and on its head it wore no other covering than a holly wreath, set here and there with shining icicles. Its dark-brown curls were long and free; free as its genial f

In [10]:
for id in results['ids'][0]:
  r = ch_collection.get(id)
  print(r['documents'][0])
  print('===================================')

“Come in!” exclaimed the Ghost. “Come in! and know me better, man!”

Scrooge entered timidly, and hung his head before this Spirit. He was not the dogged Scrooge he had been; and though the Spirit’s eyes were clear and kind, he did not like to meet them.

“I am the Ghost of Christmas Present,” said the Spirit. “Look upon me!”
“I am the Ghost of Christmas Present,” said the Spirit. “Look upon me!”

Scrooge reverently did so. It was clothed in one simple deep green robe, or mantle, bordered with white fur. This garment hung so loosely on the figure, that its capacious breast was bare, as if disdaining to be warded or concealed by any artifice. Its feet, observable beneath the ample folds of the garment, were also bare; and on its head it wore no other covering than a holly wreath, set here and there with shining icicles. Its dark-brown curls were long and free; free as its genial face, its sparkling eye, its open hand, its cheery voice, its unconstrained demeanour, and its joyful air. Gi

# Question and Answer LLM
In this exercise you will implement a question and answer LLM for the 'A Christmas Carol' book that you have chunked and saved.

The workflow is as follows:
1. Assume you ask the following question regarding the book eg. `"Who is Scrooge?"`?
2. Query the relevant context from Chroma with the question or facts from the question.
3. Combine the question and the top 5 context return by Chroma into a prompt
4. Use `google/flan-t5-base` to answer the question.

Look through the FLAN templates in [Github](https://github.com/google-research/FLAN/blob/main/flan/templates.py) and select an appropriate template for this workshop.

Do not worry about the accuracy of the result. Focus on implementing the solution. We will discuss the nuances of the solution at the end of the workshop.

Use your RAG workflow to answer the provided questions in `questions_for_rag.txt` file.

In [11]:
# TODO Your code
model_name = "google/flan-t5-base"

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [34]:
# TODO Your code
question = "Who was Scrooge's deceased business partner?"
question = "What specific, generous act does Scrooge perform for the Cratchit family on Christmas morning?"

prompt = f'{question}\n\nWhat is sentence that verbalizes this data?'
prompt_enc = tokenizer(prompt, return_tensors='pt')
answer_enc = model.generate(prompt_enc['input_ids'])
fact = tokenizer.decode(answer_enc[0], skip_special_tokens=True)

print(fact)

# Use the question to get relevant chunks
results = ch_collection.query(
    query_texts=fact,
    n_results=3
)

for k, v in results.items():
  print(f'{k}: {v}')

context = ""
for id in results['ids'][0]:
  r = ch_collection.get(id)
  context += r['documents'][0]

print(context)

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1590: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Scrooge performs a Christmas morning act for the Cratchit family.
ids: [['23bcd2c7', '0888c917', '5b2e3ada']]
embeddings: None
documents: [['“A merry Christmas, Bob!” said Scrooge, with an earnestness that could not be mistaken, as he clapped him on the back. “A merrier Christmas, Bob, my good fellow, than I have given you for many a year! I’ll raise your salary, and endeavour to assist your struggling family, and we will discuss your affairs this very afternoon, over a Christmas bowl of smoking bishop, Bob! Make up the fires and buy another coal scuttle before you dot another i, Bob Cratchit!”', '“My dear,” said Bob, “the children! Christmas Day.”\n\n“It should be Christmas Day, I am sure,” said she, “on which one drinks the health of such an odious, stingy, hard, unfeeling man as Mr. Scrooge. You know he is, Robert! Nobody knows it better than you do, poor fellow!”\n\n“My dear!” was Bob’s mild answer. “Christmas Day.”\n\n“I’ll drink his health for your sake and the day’s,” said Mrs. 

In [35]:
# TODO Your code
prompt = f'Read this: {context}\n\n{question}\nWhat is the answer? (If it cannot be answered, return "unanswerable")'
prompt = f"Read this and answer the question\n\n{context}\n\n{question}"
print(prompt)

Read this and answer the question

“A merry Christmas, Bob!” said Scrooge, with an earnestness that could not be mistaken, as he clapped him on the back. “A merrier Christmas, Bob, my good fellow, than I have given you for many a year! I’ll raise your salary, and endeavour to assist your struggling family, and we will discuss your affairs this very afternoon, over a Christmas bowl of smoking bishop, Bob! Make up the fires and buy another coal scuttle before you dot another i, Bob Cratchit!”“My dear,” said Bob, “the children! Christmas Day.”

“It should be Christmas Day, I am sure,” said she, “on which one drinks the health of such an odious, stingy, hard, unfeeling man as Mr. Scrooge. You know he is, Robert! Nobody knows it better than you do, poor fellow!”

“My dear!” was Bob’s mild answer. “Christmas Day.”

“I’ll drink his health for your sake and the day’s,” said Mrs. Cratchit, “not for his. Long life to him! A merry Christmas and a happy new year! He’ll be very merry and very happy

In [36]:
prompt_enc = tokenizer(prompt, return_tensors='pt')
answer_enc = model.generate(prompt_enc['input_ids'])
answer = tokenizer.decode(answer_enc[0], skip_special_tokens=True)

print(question)
print(answer)

What specific, generous act does Scrooge perform for the Cratchit family on Christmas morning?
raise his salary


# Discussion

1. How did your solution perform?
2. Where do you think are the issues?
3. How can you improve it?